# Cutout day-1 onboarding (demo lake)

**Environment:** uv project venv from the [Quick-start](../docs/quickstart.md) (`uv sync --extra dev`). Select kernel **Python (data-lake)**. See [notebooks.md](../docs/notebooks.md).

This notebook is the executable form of Track A in
[`docs/talks/2026-11-cutout-onboarding-plan.md`](../docs/talks/2026-11-cutout-onboarding-plan.md).
It exercises the **cutouts modality** as a first-time user on the synthetic demo lake
(`dl-demo-lake` / `ensure_demo_lake`), with a gate checklist and a friction log.

**Not for production ingest.** For your existing multi-survey lake, use
[`docs/talks/2026-11-cutout-real-lake-friction-plan.md`](../docs/talks/2026-11-cutout-real-lake-friction-plan.md).

**Success gates:** inventory → catalog `_cutout_index` → validate → `locate`/`get_cutout_at` → extract subset → (optional) ML mini-batch.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np

from data_lake.config import LakeConfig, LakeConfigNotFound
from data_lake.demo import PHOT_SURVEY, ensure_demo_lake
from data_lake.ingest.validate_cutout_ingest import run_validation
from data_lake.io.catalog import CatalogAccessor
from data_lake.io.cutouts import CutoutAccessor

# ---------------------------------------------------------------------------
# Configuration — demo lake by default
# ---------------------------------------------------------------------------
# Set USE_ENSURE_DEMO = False and point LAKE_CONFIG_TOML / DATA_LAKE_CONFIG
# at a deployment that already has DEMO_PHOT (or change SURVEY).
USE_ENSURE_DEMO = True
DEMO_DIR = Path(os.environ.get("DATA_LAKE_DEMO_DIR", "_demo_lake_cutout_day1")).expanduser().resolve()
LAKE_CONFIG_TOML: Path | None = None  # or Path("/path/to/lake_config.toml")
SURVEY = PHOT_SURVEY  # "DEMO_PHOT"
SCRATCH = Path(tempfile.mkdtemp(prefix="cutout_day1_")).resolve()
RUN_EXTRACT = True
RUN_ML_MINIBATCH = True  # needs torch (core dep)
FRICTION: list[dict] = []  # append {"gate": ..., "ok": bool, "note": ...}


def friction(gate: str, ok: bool, note: str = "") -> None:
    FRICTION.append({"gate": gate, "ok": bool(ok), "note": note})
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {gate}" + (f" — {note}" if note else ""))


print("SCRATCH:", SCRATCH)
print("SURVEY:", SURVEY)


## 0. Resolve lake root

Uses `ensure_demo_lake` (writes under `DEMO_DIR`) or `LakeConfig.discover` like the `dl-*` CLIs.


In [ ]:
if USE_ENSURE_DEMO:
    cfg_path = ensure_demo_lake(DEMO_DIR)
    os.environ["DATA_LAKE_CONFIG"] = str(cfg_path)
    os.environ.setdefault("LAKE_INGEST_TOKEN", "demo")
    print("Built/ensured demo lake config:", cfg_path)
else:
    if LAKE_CONFIG_TOML is not None:
        os.environ["DATA_LAKE_CONFIG"] = str(Path(LAKE_CONFIG_TOML).expanduser().resolve())

try:
    cfg = LakeConfig.discover(LAKE_CONFIG_TOML)
except LakeConfigNotFound as e:
    raise SystemExit(
        "No lake_config.toml found. Set USE_ENSURE_DEMO=True or export DATA_LAKE_CONFIG."
    ) from e

LAKE_ROOT = Path(cfg.lake.root).resolve()
CUTOUT_ROOT = LAKE_ROOT / "cutouts" / SURVEY
print("LAKE_ROOT:", LAKE_ROOT)
print("CUTOUT_ROOT exists:", CUTOUT_ROOT.is_dir())
friction(
    "A1_lake_resolved",
    CUTOUT_ROOT.is_dir(),
    f"cutouts/{SURVEY} missing — re-run ensure_demo_lake or pick another SURVEY",
)


## 1. Inventory (CLI + on-disk)

First-time mental model: modalities are separate trees; cutouts are Zarr under `cutouts/<survey>/`.


In [ ]:
def _run(argv: list[str]) -> subprocess.CompletedProcess[str]:
    env = os.environ.copy()
    # Prefer the kernel's environment; dl-* must be on PATH (uv env).
    return subprocess.run(argv, check=False, text=True, capture_output=True, env=env)


r1 = _run(["dl-describe-lake", "--modality", "cutouts", "--count-total"])
print(r1.stdout or r1.stderr)
ok_cli = r1.returncode == 0 and SURVEY in (r1.stdout or "")
friction("A2_describe_cutouts", ok_cli, f"returncode={r1.returncode}")

r2 = _run(["dl-preview-sources", "cutouts", SURVEY, "-n", "3"])
print(r2.stdout or r2.stderr)
friction("A2_preview_cutouts", r2.returncode == 0, f"returncode={r2.returncode}")


## 2. Catalog link (`_cutout_index`)

Demo lake patches indices for the brightest ~150 `DEMO_PHOT` sources.


In [ ]:
with CatalogAccessor(LAKE_ROOT, SURVEY) as cat:
    cols = list(cat.columns) if hasattr(cat, "columns") else []
    need = {"_source_id", "_cutout_index"}
    print("has columns:", sorted(need & set(map(str, cols))) or "(check schema another way)")
    df = cat.query(
        "SELECT _source_id, _cutout_index FROM catalog "
        "WHERE _cutout_index IS NOT NULL AND _cutout_index >= 0 LIMIT 20"
    )
    print(df)
    n = len(df)
    sample_ids = df["_source_id"].to_list() if n else []

friction("A3_catalog_cutout_index", n > 0, f"linked rows sampled={n}")
assert n > 0, "No _cutout_index >= 0 rows — demo cutout patch missing or wrong SURVEY" 


## 3. Validate cutout survey


In [ ]:
rep = run_validation(LAKE_ROOT, SURVEY, max_tiles=50)
print("errors:", rep.errors)
print("warnings:", rep.warnings)
ok = rep.ok(strict=False)
friction("A4_validate", ok, f"errors={len(rep.errors)} warnings={len(rep.warnings)}")
assert ok, "Validation failed — inspect errors above before continuing." 


## 4. Read with `CutoutAccessor` (`locate` + `get_cutout_at`)

Canonical pattern for many IDs: resolve once, then read by tile position. Close accessors when done.


In [ ]:
assert sample_ids, "Need sample_ids from gate A3"
ids = sample_ids[:8]

with CatalogAccessor(LAKE_ROOT, SURVEY) as cat, CutoutAccessor(
    LAKE_ROOT, SURVEY, catalog_accessor=cat
) as cut:
    npix, local = cut.locate(ids)
    print("npix:", npix)
    print("local:", local)
    ok_locate = int(npix[0]) >= 0 and int(local[0]) >= 0
    img, wcs = cut.get_cutout_at(int(npix[0]), int(local[0]))
    print("image shape:", getattr(img, "shape", None), "dtype:", getattr(img, "dtype", None))
    print("WCS:", wcs)

friction("A5_locate_read", ok_locate and img is not None, f"shape={getattr(img, 'shape', None)}")


## 5. Extract a small subset (CLI)

Catalog-then-cutouts via `_source_id` — same pattern as production extracts.


In [ ]:
if not RUN_EXTRACT:
    friction("A6_extract", True, "skipped (RUN_EXTRACT=False)")
else:
    id_parquet = SCRATCH / "cutout_ids.parquet"
    out_zarr = SCRATCH / "demo_cutouts_subset.zarr"
    with CatalogAccessor(LAKE_ROOT, SURVEY) as cat:
        df = cat.query(
            "SELECT _source_id FROM catalog "
            "WHERE _cutout_index IS NOT NULL AND _cutout_index >= 0 LIMIT 20"
        )
        # polars or pandas-like
        if hasattr(df, "write_parquet"):
            df.write_parquet(id_parquet)
        else:
            import pyarrow as pa
            import pyarrow.parquet as pq
            pq.write_table(pa.Table.from_pydict({"_source_id": df["_source_id"].to_list()}), id_parquet)

    r = _run([
        "dl-extract-cutout-subset",
        "--survey", SURVEY,
        "--target-list", str(id_parquet),
        "--target-id-col", "_source_id",
        "--format", "zarr",
        "--output", str(out_zarr),
    ])
    print(r.stdout or "")
    if r.returncode != 0:
        print(r.stderr)
    ok = r.returncode == 0 and out_zarr.exists()
    friction("A6_extract", ok, f"returncode={r.returncode} out={out_zarr}")


## 6. (Optional) ML mini-batch — `CutoutDataset` + `TileSequentialSampler`

Prefer tile-sequential sampling over a global shuffle on multi-tile stores.


In [ ]:
if not RUN_ML_MINIBATCH:
    friction("A7_ml_minibatch", True, "skipped (RUN_ML_MINIBATCH=False)")
else:
    try:
        from torch.utils.data import DataLoader
        from data_lake.ml.dataset import CutoutDataset, TileSequentialSampler
    except Exception as e:
        friction("A7_ml_minibatch", False, f"import failed: {e}")
    else:
        with CatalogAccessor(LAKE_ROOT, SURVEY) as cat:
            df = cat.query(
                "SELECT _source_id FROM catalog "
                "WHERE _cutout_index IS NOT NULL AND _cutout_index >= 0 LIMIT 64"
            )
            ids = np.asarray(df["_source_id"].to_list(), dtype=np.int64)
            ds = CutoutDataset(
                lake_root=LAKE_ROOT,
                survey=SURVEY,
                source_ids=ids,
                catalog_accessor=cat,
            )
        # Drop unresolved if any
        if hasattr(ds, "tile_npix"):
            keep = np.asarray(ds.tile_npix) >= 0
            n_ok = int(keep.sum())
        else:
            n_ok = len(ds)
        sampler = TileSequentialSampler(ds, shuffle_tiles=True, shuffle_within_tile=False)
        loader = DataLoader(ds, batch_size=4, sampler=sampler, num_workers=0)
        batch = next(iter(loader))
        # batch may be tensor or tuple depending on dataset
        shape = getattr(batch, "shape", None) or getattr(batch[0], "shape", None)
        print("batch type:", type(batch), "shape:", shape, "n_ok:", n_ok)
        friction("A7_ml_minibatch", n_ok > 0 and shape is not None, f"n_ok={n_ok} shape={shape}")



## 7. Gate summary / friction log

Copy failures into the real-lake friction plan when the same symptom appears on survey data.


In [ ]:
import json

print("Friction log:")
for row in FRICTION:
    print(f"  {row['gate']}: {'OK' if row['ok'] else 'FAIL'} {row.get('note','')}")

n_fail = sum(1 for r in FRICTION if not r["ok"])
summary = {
    "survey": SURVEY,
    "lake_root": str(LAKE_ROOT),
    "scratch": str(SCRATCH),
    "n_gates": len(FRICTION),
    "n_fail": n_fail,
    "friction": FRICTION,
}
out = SCRATCH / "cutout_day1_friction.json"
out.write_text(json.dumps(summary, indent=2))
print("Wrote", out)
print("RESULT:", "ALL GATES PASSED" if n_fail == 0 else f"{n_fail} gate(s) failed")
assert n_fail == 0, f"{n_fail} gate(s) failed — see FRICTION above" 


## Next steps

1. If all gates passed: open the **real-lake friction plan** and pick one catalog survey + stamp source.
2. Deeper cutout API / ingest from your own FITS: notebook [`03_cutout_ingest.ipynb`](03_cutout_ingest.ipynb).
3. Visualization / training loops: notebooks [`12_visualization.ipynb`](12_visualization.ipynb), [`13_pytorch_training_loop.ipynb`](13_pytorch_training_loop.ipynb) with `SURVEY = "DEMO_PHOT"` and `DATA_LAKE_ROOT` set to this lake's `data/` root.
